## Martian Colonist Transport -- Solution

### Data generation

In [3]:
using Distances

# create a list of all neighborhoods
neighborhoods = [:newterra, :tesla, :einstein,
    :tycho, :asimov, :galileo, :kepler,
    :hubble, :sagan, :hawking, :copernicus]

# create a list of the coordinates of each neighborhood
coords = [0 0;
    6 10;
    35 12;
    24 28;
    18 3;
    9 2;
    11 0;
    0 20;
    5 14;
    1 34;
    25 31]

# create a dictionary of coordinates with neighborhoods as indices
locations = Dict()
for i in 1:length(neighborhoods)
    locations[neighborhoods[i]] = coords[i,:]
end

# generate a list of arcs between every pair of neighborhoods
arcs = [(i,j) for i in neighborhoods for j in neighborhoods]
            
# create a dictionary of colonists currently in each neighborhood
colonist_supply = Dict(zip(neighborhoods,[14 0 17 3 8 15 20 12 2 20 21]))

# create a dictionary of colonists needed in each neighborhood
colonist_demand = Dict(zip(neighborhoods,[5 15 5 10 28 8 22 4 6 2 27]))

# calculate the distance between every pair of neighborhoods
distance = Dict()
for i in arcs
    distance[i] = evaluate(Euclidean(),locations[i[1]],locations[i[2]])
end


### Model and solution

In [12]:
using JuMP, Clp

m = Model(solver=ClpSolver())

# we have a variable on each arc representing flow of colonists
@variable(m, x[arcs] >= 0)

# at each node, meet supply/demand requirements by summing over all
# arcs leaving/entering the node
@constraint(m, supply[i in neighborhoods],sum(x[(i,j)] for j in neighborhoods) == colonist_supply[i])
@constraint(m, demand[i in neighborhoods],sum(x[(j,i)] for j in neighborhoods) == colonist_demand[i])

# 
@objective(m, Min, sum(1.75*distance[i]*x[i] for i in arcs))

solve(m)
            
println("Total cost to move colonists: ", round(getobjectivevalue(m),2))
println("Colonist movements:")
for i in arcs
    if getvalue(x[i]) > 0.00001
        println(i, ": ", getvalue(x[i]))
    end
end


Total cost to move colonists: $1601.84
Colonist movements:
(:newterra, :newterra): 5.0
(:newterra, :tesla): 6.0
(:newterra, :asimov): 1.0
(:newterra, :kepler): 2.0
(:einstein, :einstein): 5.0
(:einstein, :asimov): 12.0
(:tycho, :tycho): 3.0
(:asimov, :asimov): 8.0
(:galileo, :asimov): 7.0
(:galileo, :galileo): 8.0
(:kepler, :kepler): 20.0
(:hubble, :tesla): 8.0
(:hubble, :hubble): 4.0
(:sagan, :sagan): 2.0
(:hawking, :tesla): 1.0
(:hawking, :tycho): 7.0
(:hawking, :sagan): 4.0
(:hawking, :hawking): 2.0
(:hawking, :copernicus): 6.0
(:copernicus, :copernicus): 21.0


The cost of rehoming the colonists is 1,601.84 Martian credits. The greatest movement of colonists (12 colonists) is from Einstein to Asimov. 

## Banquet Planning -- Solution

### Data generation

In [15]:
# define a list of locations in the network
nodes = [:c, :l, :w1, :w2, :w3, :n, :h]

# create a list of all arcs that exist in the network
arcs = [(:c,:c),(:c,:w1),(:c,:w2),(:c,:w3),(:c,:n),(:c,:h),
        (:l,:l),(:l,:w1),(:l,:w2),(:l,:w3),(:l,:n,),(:l,:h),
        (:w1,:w1),(:w1,:w2),(:w1,:w3),(:w1,:n),(:w1,:h),
        (:w2,:w1),(:w2,:w2),(:w2,:w3),(:w2,:n),(:w2,:h),
        (:w3,:w1),(:w3,:w2),(:w3,:w3),(:w3,:n),(:w3,:h),
        (:n,:n),(:n,:h),(:h,:n),(:h,:h)]

# create a Dictionary of the demand at each location (indexed over nodes)
# (positive for source nodes, negative for sink nodes, 0 everywhere else)
demand = Dict(zip(nodes,[160000, 150000, 0, 0, 0, -200000, -110000]))

# create a Dictionary of the cost of transporting 1000 napkins
# (indexed over arcs)
cost = Dict(zip(arcs,[0 12 11 10 30 25 0 9 12 15 35 22 0 5 8 10 17 8 0 6 14 12 5 9 0 11 15 0 18 18 0]))
;


### Model

In [18]:
using JuMP, Clp

m = Model(solver=ClpSolver())

# create a variable indexed over arcs
@variable(m, x[arcs] >= 0)

# create balance constraints for each node (flow out - flow in = demand)
@constraint(m, constr[n in nodes], sum(x[j] for j in arcs if j[1] == n) - sum(x[(j)] for j in arcs if j[2] == n) == demand[n])

# minimize cost of transporting each bunch of 1,000 napkins
@objective(m, Min, sum(0.001*cost[i]*x[i] for i in arcs))
                            
solve(m)
                            
println("Total cost of meeting demand: \$", getobjectivevalue(m))
println("Shipments: ")
for i in arcs
    if getvalue(x[i]) > 0.00001
        println(i, ": ", getvalue(x[i]))
    end
end

Total cost of meeting demand: $6430.0
Shipments: 
(:c, :w2): 110000.0
(:c, :w3): 50000.0
(:l, :w1): 150000.0
(:w1, :n): 150000.0
(:w2, :h): 110000.0
(:w3, :n): 50000.0


We meet demand at a cost of \$6,430. To do so, we ship 110,000 napkins from Chicago to warehouse 2, 50,000 napkins from Chicago to warehouse 3, and 150,000 napkins from LA to warehouse 1. Then we ship all 150,000 napkins from warehouse 1 and 50,000 napkins from warehouse 3 to New York and 110,000 napkins from warehouse 2 to Houston.

## The Duality of Man('s Diet) -- Solution

### Part (a)

In [33]:
# We can reuse the code from homework 2 for this problem
using DataFrames, CSV, NamedArrays 

#Load the data file (ref: Boyd/263)
raw = CSV.read("stigler.csv");

# turn DataFrame into array
diet_array = convert(Array,raw); 

# the names of the DataFrame (header) are nutrients
nutrients = names(raw[2:end]);

# create a list of foods from the diet array
foods = diet_array[2:end,1];

# create a list of the minimum requirements of each nutrient
min_req = Dict(zip(nutrients,diet_array[1,2:end]));

using NamedArrays
recipe_matrix = diet_array[2:end,2:end]
recipe = NamedArray(recipe_matrix, (foods, nutrients), ("foods","nutrients"))

using JuMP, Clp

m = Model(solver=ClpSolver())

# create variables representing amount of each food to eat each day
@variable(m, x[foods] >= 0) 

# objective is to minimize the cost/day of meeting diet requirements
@objective(m, Min, sum(x[t] for t in foods))

# for each nutrient, we must eat at least the recommended daily amount
@constraint(m, constr[i in nutrients], sum(recipe[t,i] * x[t] for t in foods) >= min_req[i] );

solve(m)
# to determine how much we will pay for calcium and protein, we 
# get the shadow prices of those constraints (dual variable values)
println("We would pay up to \$ ", getdual(constr[Symbol("Protein (g)")]), " for a gram of Protein")

println("We would pay up to \$ ", round(getdual(constr[Symbol("Calcium (g)")]),2), " for a gram of Calcium")


We would pay up to $ 0.0 for a gram of Protein
We would pay up to $ 0.03 for a gram of Calcium


Since a Calcium pill contains 0.5 grams of Calcium, we would only be willing to pay \$0.015 for a Calcium supplement.

### Part (b)

We can add a row to the stigler.csv file that has 0s in all nutrient colums except Calcium. Since we have to normalize it to be the nutrients in \$1's worth of food, we multiply 0.5*100 and get 50 grams of Calcium from one dollar of Calcium supplements.


In [36]:
#Load the data file (ref: Boyd/263)
raw = CSV.read("stigler.csv");

# turn DataFrame into array
diet_array = convert(Array,raw); 

# the names of the DataFrame (header) are nutrients
nutrients = names(raw[2:end]);

# create a list of foods from the diet array
foods = diet_array[2:end,1];

# create a list of the minimum requirements of each nutrient
min_req = Dict(zip(nutrients,diet_array[1,2:end]));

recipe_matrix = diet_array[2:end,2:end]
recipe = NamedArray(recipe_matrix, (foods, nutrients), ("foods","nutrients"))

m = Model(solver=ClpSolver())

# create variables representing amount of each food to eat each day
@variable(m, x[foods] >= 0) 

# objective is to minimize the cost/day of meeting diet requirements
@objective(m, Min, sum(x[t] for t in foods))

# for each nutrient, we must eat at least the recommended daily amount
@constraint(m, constr[i in nutrients], sum(recipe[t,i] * x[t] for t in foods) >= min_req[i] );

solve(m)

println("New optimal cost: \$", round(365.25*getobjectivevalue(m),2))
println("New optimal annual diet: ")
for i in foods
    if getvalue(x[i]) > 0.00001
        println(i, ": ", 365.25*getvalue(x[i]))
    end
end

New optimal cost: $37.02
New optimal annual diet: 
Wheat Flour (Enriched): 24.099415274648024
Liver (Beef): 2.8651447909687167
Cabbage: 4.088983842757778
Spinach: 1.428600629029006
Calcium: 4.541444102429581


## Alien Council -- Solution

Note: There might be many correct ways to model this problem as a max flow problem. I am showing one such way, but if you find a solution that meets the assignment requirements as a max flow problem in another way, that's fine!

### Data generation

In [13]:
# species on the council
species = [:human, :ildiran, :verdani, :roamer]

# all the possible candidates for the council
candidates = [:basil_wenceslas, :king_peter, :eldred_cain, :prince_daniel,
    :kurt_lanyan, :patrick_fitzpatrick, :anton_colicos, :sarein, :estarra,
    :beneto, :otema, :nira, :celli, :solimar, :nahton, :yarrod, :kolker, :rossia,
    :ross_tamblyn, :jess_tamblyn, :tasia_tamblyn, :cesca_peroni, :jhy_okiah,
    :del_dellum, :zhett_kellum, :niko_chan_tylar,:cyroch, :jorah, :urdruh,
    :rusah, :thorh, :yazrah, :peryh, :osirah, :korinh, :zannh, :ohn, :vaosh]

# candidates that are human
human_candidates = [:basil_wenceslas, :king_peter, :eldred_cain, :prince_daniel,
    :kurt_lanyan, :patrick_fitzpatrick,
    :anton_colicos, :sarein, :estarra]

# candidates that are verdani
verdani_candidates = [:beneto, :otema, :nira, :celli, :solimar,
    :nahton, :yarrod, :kolker, :rossia]

# candidates that are roamers
roamer_candidates = [:ross_tamblyn, :jess_tamblyn, :tasia_tamblyn,
    :cesca_peroni, :jhy_okiah, :del_dellum, :zhett_kellum, :niko_chan_tylar]

# candidates that are ildiran
ildiran_candidates = [:cyroch, :jorah, :urdruh, :rusah, :thorh, :yazrah,
    :peryh, :osirah, :korinh, :zannh, :ohn, :vaosh]

# subject areas needed on the council
subject_areas = [:physics, :astronomy, :logistics, :military, :history,
    :literature, :art, :biology, :chemistry, :politics, :mathematics,
    :philosophy, :archeology, :theology, :anthropology, :engineering]

# set a seed so we get the same random numbers every time
seed = srand(29828954)

# randomly match candidates to subject areas
subject_candidate = Dict()
density = 0.1 # lower density means matches less likely
for c in candidates
    for s in subject_areas
        if c == :ross_tamblyn
#         # if a uniform(0,1) number is less than density
#         if rand() < density
#         # the candidate is an expert in that area
            subject_candidate[(s,c)] = 1
        else
            subject_candidate[(s,c)] = 0
        end
    end
end
# we need exactly one candidate per subject area
num_from_subject = Dict(zip(subject_areas,ones(length(subject_areas))))

# we can have at most 4 council members from each species
max_from_species = Dict(zip(species,4*ones(length(species))));

# create the list of nodes in the graph
nodes = vcat([:source], subject_areas, candidates, species, [:sink])

# create all arcs needed in the network
arcs_1 = [(:source,i) for i in subject_areas] # arcs from source to subjects
arcs_2 = [n[1] for n in subject_candidate if subject_candidate[n[1]]==1] # arcs from subjects to experts in that subject
arcs_3 = [(i,:human) for i in candidates if i in human_candidates] # arcs from humans to "human" node
arcs_4 = [(i,:ildiran) for i in candidates if i in ildiran_candidates] # arcs from ildirans to "ildiran" node
arcs_5 = [(i,:roamer) for i in candidates if i in roamer_candidates] # arcs from roamers to "roamer" node
arcs_6 = [(i,:verdani) for i in candidates if i in verdani_candidates] # arcs from verdani to "verdani" node
                                                            
arcs_1 = [(:source,i) for i in species] # arcs from source to subjects
arcs_3 = [(:human,i) for i in candidates if i in human_candidates] # arcs from humans to "human" node
arcs_4 = [(:ildiran,i) for i in candidates if i in ildiran_candidates] # arcs from ildirans to "ildiran" node
arcs_5 = [(:roamer,i) for i in candidates if i in roamer_candidates] # arcs from roamers to "roamer" node
arcs_6 = [(:verdani,i) for i in candidates if i in verdani_candidates] # arcs from verdani to "verdani" node
     
arcs_2 = [n[1] for n in subject_candidate if subject_candidate[n[1]]==1] # arcs from subjects to experts in that subject
                                                                                                                   
println(arcs_2)
# concatenate all the arcs and add arcs from each species to the sink
# also add the dummy arc from sink to source
arcs = vcat(arcs_1, arcs_2, arcs_3, arcs_4, arcs_5, arcs_6, 
                [(:human,:sink),(:verdani,:sink),(:roamer,:sink),(:ildiran,:sink)], 
                                                    [(:sink,:source)])
# create a dictionary of capacities indexed over arcs                           
capacity = Dict()
for a in arcs
    # capacity on each arc to the subject areas is 1 (if max flow < 16, no solution exists)
    if a[1] == :source
        capacity[a] = 4
    # capacity on each arc between subjects and candidates is 1 (assignment can only be made once)
    elseif a[1] in species
        capacity[a] = 1
    # capacity on each arc between candidates and species is upper bound on total flow 
    # (someone can be an expert in multiple subjects)
    elseif a[1] in candidates
        capacity[a] = 1
    # capacity on each arc between species and sink is 4
    elseif a[1] in subject_areas
        capacity[a] = 1
    # capacity on dummy arc is an upper bound on total flow
    elseif a == (:sink,:source)
        capacity[a] = 16
    end
end
println(capacity)                                               

# create a dictionary of costs, indexed over arcs
cost = Dict()
for a in arcs
    # dummy arc cost is 1
    if a == (:sink,:source)
        cost[a] = -1
    # all other arc costs are 0
    else
        cost[a] = 0
    end
end


Tuple{Symbol,Symbol}[(:theology, :ross_tamblyn), (:philosophy, :ross_tamblyn), (:chemistry, :ross_tamblyn), (:anthropology, :ross_tamblyn), (:astronomy, :ross_tamblyn), (:mathematics, :ross_tamblyn), (:politics, :ross_tamblyn), (:physics, :ross_tamblyn), (:archeology, :ross_tamblyn), (:history, :ross_tamblyn), (:engineering, :ross_tamblyn), (:literature, :ross_tamblyn), (:biology, :ross_tamblyn), (:military, :ross_tamblyn), (:art, :ross_tamblyn), (:logistics, :ross_tamblyn)]
Dict{Any,Any}(Pair{Any,Any}((:mathematics, :ross_tamblyn), 1),Pair{Any,Any}((:ildiran, :ohn), 1),Pair{Any,Any}((:ildiran, :korinh), 1),Pair{Any,Any}((:theology, :ross_tamblyn), 1),Pair{Any,Any}((:human, :eldred_cain), 1),Pair{Any,Any}((:human, :anton_colicos), 1),Pair{Any,Any}((:human, :basil_wenceslas), 1),Pair{Any,Any}((:ildiran, :zannh), 1),Pair{Any,Any}((:engineering, :ross_tamblyn), 1),Pair{Any,Any}((:verdani, :beneto), 1),Pair{Any,Any}((:ildiran, :rusah), 1),Pair{Any,Any}((:philosophy, :ross_tamblyn), 1),Pair

### Model

In [12]:
using JuMP, Clp 
                                                
m = Model(solver=ClpSolver())

# create a variable representing flow on each arc
@variable(m,  x[arcs] >= 0)

# capacity constraints on each arc
@constraint(m, cap[a in arcs], x[a] <= capacity[a])

# balance constraints (all demand = 0, so flow in = flow out)
@constraint(m, constr[n in nodes], sum(x[a] for a in arcs if a[1] == n) == sum(x[a] for a in arcs if a[2] == n))

# maximize flow on dummy arc
@objective(m, Max, x[(:sink,:source)])
                            
solve(m)

if getobjectivevalue(m) == 16.0                                                                        
    println("Solution found!" )
    println("Assignments: ")
    for i in arcs
        if getvalue(x[i]) > 0.00001
            println(i, ": ", getvalue(x[i]))
        end
    end
else
    println("No solution found" )
end
                            

No solution found
